# Screening Decisions Analysis

Analyze the paper screening decisions made by each rater and export the final included papers as a BibTeX file.

In [17]:
import re
from pathlib import Path
import os

import bibtexparser
import pandas as pd

from sklearn.metrics import cohen_kappa_score

## Screening Decisions

Load each rater's decisions and inspect the include/exclude distribution.

In [2]:
lucas_df = pd.read_csv("../decisions/lucas.csv", usecols=["id", "decision", "reason"])
lucas_df.head()

,id,decision,reason
0,52,include,IC1
1,60,include,IC2
2,71,include,IC2
3,77,include,IC1
4,107,exclude,EC2


In [3]:
lucas_df["decision"].value_counts()

decision
exclude    211
include     33
Name: count, dtype: int64

In [4]:
victor_df = pd.read_csv("../decisions/victor.csv", usecols=["id", "decision", "reason", "notes"], dtype={"notes": str})
victor_df.head()

,id,decision,reason,notes
0,52,exclude,EC2,NaN
1,60,include,IC1,NaN
2,71,include,IC2,NaN
3,77,include,IC1,NaN
4,107,exclude,EC2,NaN


In [5]:
victor_df["decision"].value_counts()

decision
exclude    160
include     84
Name: count, dtype: int64

In [6]:
kappa = cohen_kappa_score(lucas_df["decision"], victor_df["decision"])
print(f"Cohen's Kappa: {kappa:.3f}")

Cohen's Kappa: 0.374


> What magnitude of kappa reflects adequate agreement?

There are many guidelines that tries to answer this question, but any set of guidelines is however by no means universally accepted (usually arbitrary and based on personal opinion).

Anyways, Fleiss's guidelines characterize kappas over 0.75 as excellent, 0.40 to 0.75 as fair to good, and below 0.40 as poor. We can say that our agreement is almost **fair**.

In [7]:
all_papers_df = pd.read_csv("../artifacts/all_papers.csv", usecols=["canonical_id","title","abstract"])
all_papers_df.head()

,canonical_id,title,abstract
0,1,Sustainability and\&nbsp;High Performance Comp...,The use of High Performance Computing (HPC) ha...
1,2,Environment-conscious scheduling of HPC applic...,The use of High Performance Computing (HPC) in...
2,3,A Digital Twin Framework for Liquid-cooled Sup...,"We present ExaDigiT, an open-source framework ..."
3,4,Toward Sustainable HPC: In-Production Deployme...,This paper describes the deployment and operat...
4,5,Designing an Energy-Efficient HPC Supercomputi...,This paper presents design considerations that...


In [8]:
conflicts_df = lucas_df[lucas_df["decision"] != victor_df["decision"]].copy()
conflicts_df.rename(columns={"decision": "lucas_decision", "reason": "lucas_reason"}, inplace=True)

conflicts_df["victor_decision"] = victor_df.loc[conflicts_df.index, "decision"]
conflicts_df["victor_reason"] = victor_df.loc[conflicts_df.index, "reason"]
conflicts_df["victor_notes"] = victor_df.loc[conflicts_df.index, "notes"]

conflicts_df = pd.merge(conflicts_df, all_papers_df, left_on="id", right_on="canonical_id")

conflicts_df = conflicts_df[["id", "title", "abstract", "lucas_decision", "lucas_reason", "victor_decision", "victor_reason", "victor_notes"]]

conflicts_df["final_decision"] = ""
conflicts_df["resolution"] = ""
conflicts_df.head()

,id,title,abstract,lucas_decision,lucas_reason,victor_decision,victor_reason,victor_notes,final_decision,resolution
0,52,The Power of\&nbsp;Training: How Different Neu...,This work offers a heuristic evaluation of the...,include,IC1,exclude,EC2,NaN,,
1,223,Data Center Operators Face Energy Irony,High-performance computational technology is e...,exclude,EC1,include,IC1,NaN,,
2,237,"Sustainable AI: Experiences, Challenges \&amp;...",The use of Artificial Intelligence (AI) and Ma...,exclude,EC2,include,IC1,NaN,,
3,408,Empowering Generative AI in Enterprises: Susta...,Rapid growth in unstructured data has triggere...,exclude,EC2,include,IC1,NaN,,
4,445,A deep dive into sustainable generative AI and...,The lecture covers the foundations of sustaina...,exclude,EC1,include,IC1,NaN,,


In [9]:
conflicts_csv_path = "../artifacts/conflicts.csv"

if not os.path.exists(conflicts_csv_path):
    print(f"Exporting conflicts to {conflicts_csv_path}...")
    conflicts_df.to_csv(conflicts_csv_path, index=False)
else:
    print(f"Conflicts file already exists at {conflicts_csv_path}....")

Conflicts file already exists at ../artifacts/conflicts.csv....


### Merging decisions

In [10]:
if os.path.exists(conflicts_csv_path):
    print(f"Conflicts file already exists at {conflicts_csv_path}, assuming it has been resolved. Loading resolved conflicts...")
    conflicts_df = pd.read_csv(conflicts_csv_path)
else:
    print(f"No conflicts file found at {conflicts_csv_path}, assuming conflicts have not been resolved yet.")

Conflicts file already exists at ../artifacts/conflicts.csv, assuming it has been resolved. Loading resolved conflicts...


In [11]:
first_df = pd.read_csv("../decisions/first.csv")
first_df.head()

,id,decision,reason
0,1,include,IC1
1,2,include,IC1
2,3,include,IC1
3,4,include,IC1
4,5,include,IC1


In [12]:
# The dataframe conflicts_df contains all papers where there was a disagreement between Lucas and Victor. They were resolved by discussion, and the final decisions were recorded in the "final_decision" column.
# Use lucas_df as the source of truth for all papers where there was agreement, and use conflicts_df for the papers where there was disagreement, taking the final decision from the "final_decision" column. Combine these into a single dataframe with all papers and their final decisions.
final_decisions_df = pd.merge(lucas_df[["id", "decision"]], conflicts_df[["id", "final_decision"]], on="id", how="outer")
final_decisions_df.head()

,id,decision,final_decision
0,52,include,exclude
1,60,include,NaN
2,71,include,NaN
3,77,include,NaN
4,107,exclude,NaN


In [13]:
final_decisions_df["final_decision"] = final_decisions_df.apply(
    lambda row: row["decision"] if pd.isna(row["final_decision"]) else row["final_decision"],
    axis=1
)
final_decisions_df.drop(columns=["decision"], inplace=True)
final_decisions_df.rename(columns={"final_decision": "decision"}, inplace=True)
final_decisions_df.head()

,id,decision
0,52,exclude
1,60,include
2,71,include
3,77,include
4,107,exclude


In [14]:
all_decisions = pd.concat([first_df, final_decisions_df])
all_decisions["decision"].value_counts()

decision
exclude    547
include     62
Name: count, dtype: int64

In [15]:
to_include_df = final_decisions_df[final_decisions_df["decision"] == "include"]
to_include_df.head()

,id,decision
1,60,include
2,71,include
3,77,include
13,188,include
14,223,include


## BibTeX Export

Load the full BibTeX library and `artifacts/all_papers.csv`, match each included paper by title, and write the results to `artifacts/included.bib`.

In [18]:
with open("../papers/papers.bib", encoding="utf-8") as f:
    library = bibtexparser.loads(f.read())

In [19]:
papers_df = pd.read_csv("../artifacts/all_papers.csv", usecols=["id", "canonical_id", "title", "abstract"])
included_papers = papers_df[papers_df["id"].isin(to_include_df["id"])]
print(f"Papers to include: {len(included_papers)}")
included_papers.head()

Papers to include: 32


,id,canonical_id,title,abstract
59,60,60,The Energy Efficiency Research of\&nbsp;Code f...,Last ten years the top performance of the fast...
70,71,71,What A Waste,The immense demand for high performance comput...
76,77,77,Adaptive Carbon-Aware Scheduling Policies for\...,In response to growing energy costs and carbon...
187,188,188,Enabling distributed generation powered sustai...,The necessity for capping carbon emission has ...
222,223,223,Data Center Operators Face Energy Irony,High-performance computational technology is e...


In [20]:
import re

def normalize_title(t):
    # Collapse \cmd{arg} -> \cmdarg so CSV and bib titles compare equal
    return re.sub(r'\{(\w+)\}', r'\1', t).strip()

title_to_entry = {
    normalize_title(e["title"]): e
    for e in library.entries
    if "title" in e
}

included_entries = []
missing = []
for _, row in included_papers.iterrows():
    entry = title_to_entry.get(normalize_title(row["title"]))
    if entry:
        included_entries.append(entry)
    else:
        missing.append(row["id"])

print(f"Found: {len(included_entries)}, Missing: {len(missing)}")
if missing:
    print("Missing IDs:", missing)

Found: 32, Missing: 0


In [21]:
included_bib_path = "../artifacts/included.bib"

out_lib = bibtexparser.bibdatabase.BibDatabase()
out_lib.entries = included_entries

bibtex_str = bibtexparser.dumps(out_lib)
with open(included_bib_path, "w", encoding="utf-8") as f:
    f.write(bibtex_str)

print(f"Written {len(included_entries)} entries to {included_bib_path}")

Written 32 entries to ../artifacts/included.bib
